# HydroSAR-BD: Complete Replication Notebook

**Spatiotemporal Gaussian Mixture Model (ST-GMM) for Dynamic Surface Water Mapping in Bangladesh**

This notebook reproduces **every result** of the HydroSAR-BD manuscript end-to-end:

| Section | Description |
|:---|:---|
| **1** | Setup & dependency installation |
| **2** | ST-GMM threshold calibration from SAR backscatter histograms |
| **3** | Surface water area computation (district → national) |
| **4** | GMM component justification — AIC/BIC information criteria |
| **5** | Per-class accuracy assessment (Permanent / Semi-permanent / Ephemeral) |
| **6** | Comparative model benchmark (Random Forest, Otsu, ST-GMM, NDWI) |
| **7** | Publication figures (seasonal ribbon, July peak trend, divisional heatmap) |

> **Data note:** Place the required CSV files in the `data/` folder before running.  
> All GEE export scripts are in `gee_scripts/`.


## Section 1 — Setup & Dependencies

In [ ]:
import subprocess, sys
pkgs = ["pandas", "numpy", "scikit-learn", "scipy", "matplotlib", "rasterio"]
for p in pkgs:
    try:
        __import__(p.replace("-","_").replace("scikit_learn","sklearn"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

import os, ast, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.mixture import GaussianMixture
from scipy import stats as scipy_stats
from scipy.stats import norm
warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(DATA_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

HIST_CSV  = os.path.join(DATA_DIR, "Bangladesh_District_VV_Histograms_2015_2025.csv")
OCCUR_CSV = os.path.join(DATA_DIR, "Validation_Points_With_Occurrence.csv")
GEO_CSV   = os.path.join(DATA_DIR, "GEE_Upload_Ready_LatLon.csv")

MONTH_NAMES = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
               7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
print("✓ Environment ready.")


## Section 2 — ST-GMM Threshold Calibration

The core algorithm fits a **2-component Gaussian Mixture Model** to the Sentinel-1 VV backscatter histogram for each district-month pair.  
The water/land decision boundary is located at the intersection of the two Gaussian components.


In [ ]:
def fit_gmm_threshold(counts, bins):
    mask = counts > 0
    counts, bins = counts[mask], bins[mask]
    if len(bins) < 5 or counts.sum() < 100:
        return np.nan
    samples = np.repeat(bins, counts.astype(int)).reshape(-1, 1)
    try:
        gmm = GaussianMixture(n_components=2, covariance_type='full',
                              max_iter=200, random_state=42)
        gmm.fit(samples)
        means  = gmm.means_.flatten()
        stds   = np.sqrt(gmm.covariances_.flatten())
        weights= gmm.weights_.flatten()
        idx    = np.argsort(means)
        means, stds, weights = means[idx], stds[idx], weights[idx]
        x      = np.linspace(bins.min(), bins.max(), 1000)
        pdf_w  = weights[0] * norm.pdf(x, means[0], stds[0])
        pdf_l  = weights[1] * norm.pdf(x, means[1], stds[1])
        mask_s = (x > means[0]) & (x < means[1])
        if not mask_s.any():
            return float((means[0]*stds[1]+means[1]*stds[0])/(stds[0]+stds[1]))
        diff   = pdf_w[mask_s] - pdf_l[mask_s]
        sc     = np.where(np.diff(np.sign(diff)))[0]
        return float(x[mask_s][sc[0]]) if len(sc) else                float((means[0]*stds[1]+means[1]*stds[0])/(stds[0]+stds[1]))
    except Exception:
        return np.nan

if not os.path.exists(HIST_CSV):
    print(f"[MISSING] {HIST_CSV}")
else:
    print("Loading histogram CSV …")
    df_hist = pd.read_csv(HIST_CSV)
    df_hist['hist'] = df_hist['histogram_counts'].apply(ast.literal_eval)
    if 'histogram_means' in df_hist.columns:
        df_hist['bins'] = df_hist['histogram_means'].apply(ast.literal_eval)
    else:
        df_hist['bins'] = [np.linspace(-30, 5, len(h)) for h in df_hist['hist']]

    print(f"  Loaded {len(df_hist):,} district-month rows")

    df_hist['threshold'] = df_hist.apply(
        lambda r: fit_gmm_threshold(np.array(r['hist']), np.array(r['bins'])), axis=1)

    n_fail = df_hist['threshold'].isna().sum()
    print(f"  GMM converged: {len(df_hist)-n_fail}/{len(df_hist)} | Failed: {n_fail}")

    lookup = df_hist.groupby(['district_name','month'])['threshold'].mean().reset_index()
    lookup.to_csv(os.path.join(RESULTS_DIR,"GMM_Threshold_Lookup.csv"), index=False)
    print(f"✓ Threshold lookup saved  →  results/GMM_Threshold_Lookup.csv")
    print(lookup.head(10).to_string(index=False))


## Section 3 — Surface Water Area Computation

Apply the district-month GMM threshold to each histogram to count water pixels.  
Pixel scale = 100 m → pixel area = 0.01 km².  Missing entries are filled by linear temporal interpolation.


In [ ]:
PIXEL_AREA_KM2 = (100**2) / 1e6   # 0.01 km2

def compute_water_km2(bin_centers, counts, threshold):
    bc, ct = np.array(bin_centers), np.array(counts)
    n = min(len(bc), len(ct))
    return float(np.sum(ct[:n][bc[:n] <= threshold])) * PIXEL_AREA_KM2

if os.path.exists(HIST_CSV) and 'lookup' in dir():
    thresh_lookup = {(r.district_name, r.month): r.threshold
                     for _, r in lookup.iterrows()}
    national_fallback = df_hist.groupby('month')['threshold'].mean().to_dict()

    area_records = []
    for _, row in df_hist.iterrows():
        bc = np.array(row['bins']); ct = np.array(row['hist'])
        th = thresh_lookup.get((row['district_name'], row['month']),
             national_fallback.get(row['month'], -12.0))
        if pd.isna(th): continue
        area_records.append({'year': row['year'], 'month': row['month'],
                             'district': row['district_name'],
                             'water_area_km2': compute_water_km2(bc, ct, th)})

    area_df = pd.DataFrame(area_records)
    national = area_df.groupby(['year','month'])['water_area_km2'].sum().reset_index()
    national.to_csv(os.path.join(RESULTS_DIR,"national_monthly_water_area.csv"), index=False)
    print(f"✓ National water area saved  ({len(national)} rows)")

    # Quick summary
    peak = national.loc[national['water_area_km2'].idxmax()]
    low  = national.loc[national['water_area_km2'].idxmin()]
    print(f"  Peak  : {peak.water_area_km2:,.0f} km²  (Year {int(peak.year)}, Month {int(peak.month)})")
    print(f"  Trough: {low.water_area_km2:,.0f} km²  (Year {int(low.year)}, Month {int(low.month)})")
else:
    print("[SKIPPED] Histogram CSV not available")


## Section 4 — GMM Component Justification (AIC / BIC)

We compare 2, 3, 4, and 5-component GMMs across three geographically diverse districts.  
Lower AIC/BIC indicates better model parsimony. Physical interpretability (water vs. non-water) favours the 2-component model.


In [ ]:
sample_districts = ['Sunamganj', 'Dhaka', 'Bhola']

if os.path.exists(HIST_CSV):
    aic_results = []
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, dist in zip(axes, sample_districts):
        sub = df_hist[df_hist['district_name'] == dist]
        sub = sub[sub['month'] == 8] if not sub[sub['month']==8].empty else sub
        row = sub.iloc[0]
        bc  = np.array(row['bins']); ct = np.array(row['hist'])
        mask = ct > 0
        total = ct[mask].sum()
        scale = max(1, int(total // 100000))
        scaled= (ct[mask] / scale).astype(int)
        samples = np.repeat(bc[mask], scaled).reshape(-1, 1)

        n_range = [2, 3, 4, 5]
        aic_s, bic_s = [], []
        for n in n_range:
            g = GaussianMixture(n_components=n, covariance_type='full',
                                max_iter=300, random_state=42).fit(samples)
            aic_s.append(g.aic(samples))
            bic_s.append(g.bic(samples))
            aic_results.append({'District': dist, 'Components': n,
                                 'AIC': g.aic(samples), 'BIC': g.bic(samples)})

        ax.plot(n_range, aic_s, 'o-', lw=2, label='AIC')
        ax.plot(n_range, bic_s, 's--', lw=2, label='BIC')
        ax.axvline(2, color='red', lw=1.5, ls=':', alpha=0.6, label='Selected (n=2)')
        ax.set_title(dist, fontsize=13, fontweight='bold')
        ax.set_xlabel('Number of GMM Components')
        ax.set_ylabel('Information Criterion Score')
        ax.set_xticks(n_range)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, ls='--')

    plt.suptitle('GMM Goodness-of-Fit Diagnostics — AIC & BIC', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR,'fig_gmm_aic_bic.png'), dpi=300)
    plt.show()

    aic_df = pd.DataFrame(aic_results)
    aic_df.to_csv(os.path.join(RESULTS_DIR,'GMM_AIC_BIC_Scores.csv'), index=False)
    print("✓ AIC/BIC figure saved  →  figures/fig_gmm_aic_bic.png")
    print(aic_df.to_string(index=False))
else:
    print("[SKIPPED] Histogram CSV not available")


## Section 5 — Per-Class Accuracy Assessment

Ground-truth points are stratified by **JRC Global Surface Water occurrence frequency** into three hydroperiod classes:

| Class | Occurrence frequency |
|:---|:---|
| Permanent water | ≥ 80% |
| Semi-permanent water | 40% – 79% |
| Ephemeral water | 1% – 39% |

**User's Accuracy (UA)** = TP / (TP + FP) — of all pixels classified as water, how many are actually water?  
**Producer's Accuracy (PA)** = TP / (TP + FN) — of all true water pixels, how many were correctly detected?


In [ ]:
if not os.path.exists(OCCUR_CSV):
    print(f"[MISSING] {OCCUR_CSV}")
    print("Run gee_scripts/04_extract_jrc_occurrence.js in GEE and place the CSV in data/")
else:
    df_val = pd.read_csv(OCCUR_CSV)
    df_val['occurrence'] = df_val['occurrence'].fillna(0)

    def hydroperiod(occ):
        if occ >= 80:   return 'Permanent'
        elif occ >= 40: return 'Semi-permanent'
        elif occ > 0:   return 'Ephemeral'
        else:           return 'Non-water'

    df_val['Water_Class'] = df_val['occurrence'].apply(hydroperiod)

    rows = []
    for cls in ['Permanent', 'Semi-permanent', 'Ephemeral']:
        sub = df_val[df_val['Water_Class'] == cls]
        if len(sub) == 0:
            print(f"  [WARNING] No points in class: {cls}")
            continue
        y_true = sub['Field_Truth']; y_pred = sub['class']
        tp = ((y_true==1)&(y_pred==1)).sum()
        fp = ((y_true==0)&(y_pred==1)).sum()
        fn = ((y_true==1)&(y_pred==0)).sum()
        tn = ((y_true==0)&(y_pred==0)).sum()
        ua = (tp/(tp+fp)*100) if (tp+fp)>0 else 0.0
        pa = (tp/(tp+fn)*100) if (tp+fn)>0 else 0.0
        oa = ((tp+tn)/len(sub)*100)
        rows.append({'Water Class': cls, 'N': len(sub),
                     'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                     "User's Accuracy (%)": round(ua,2),
                     "Producer's Accuracy (%)": round(pa,2),
                     "Class Accuracy (%)": round(oa,2)})

    acc_df = pd.DataFrame(rows)
    acc_df.to_csv(os.path.join(RESULTS_DIR,'per_class_accuracy.csv'), index=False)

    print("=" * 68)
    print("         PER-CLASS ACCURACY METRICS")
    print("=" * 68)
    print(acc_df.to_string(index=False))
    print("=" * 68)
    print("✓ Saved  →  results/per_class_accuracy.csv")


## Section 6 — Comparative Model Benchmark

This section plots the 5-panel comparative map of water classification methods using the exported GeoTIFF rasters.  
Rasters must be placed in `data/rasters/` after running `gee_scripts/03_export_comparative_models.js`.


In [ ]:
try:
    import rasterio
    RASTER_DIR = os.path.join(DATA_DIR, "rasters")
    raster_files = {
        '(a) SAR VV\n(Raw Backscatter)':    os.path.join(RASTER_DIR,'SAR_VV.tif'),
        '(b) Random Forest\n(Supervised)':  os.path.join(RASTER_DIR,'RandomForest.tif'),
        '(c) Otsu\n(Global Unsupervised)':  os.path.join(RASTER_DIR,'Otsu.tif'),
        '(d) ST-GMM\n(Proposed Method)':    os.path.join(RASTER_DIR,'ST_GMM.tif'),
        '(e) Sentinel-2 NDWI\n(Optical Reference)': os.path.join(RASTER_DIR,'NDWI.tif'),
    }
    missing = [f for f in raster_files.values() if not os.path.exists(f)]
    if missing:
        print("[INFO] Raster files not found. Run gee_scripts/03_export_comparative_models.js in GEE first.")
        print("Expected files in data/rasters/:")
        for f in raster_files.values(): print(" ", os.path.basename(f))
    else:
        cmap_binary = matplotlib.colors.ListedColormap(['#e0e0e0','#004c99'])
        fig, axes = plt.subplots(1, 5, figsize=(25, 6))
        for ax, (title, fpath) in zip(axes, raster_files.items()):
            with rasterio.open(fpath) as src:
                img = src.read(1)
                img = np.ma.masked_where(img < -9999, img)
            if 'SAR' in title:
                ax.imshow(img, cmap='gray', vmin=-25, vmax=0)
            else:
                ax.imshow(img, cmap=cmap_binary, vmin=0, vmax=1)
            ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
            ax.axis('off')
        plt.suptitle('Comparative Water Mapping — Study Area, Bangladesh', fontsize=14, fontweight='bold')
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR,'fig_comparative_5panel.png'), dpi=300)
        plt.show()
        print("✓ Figure saved  →  figures/fig_comparative_5panel.png")
except ImportError:
    print("[INFO] rasterio not installed. Run: pip install rasterio")


## Section 7 — Publication Figures

Generates the manuscript figures from the computed water area time series.


In [ ]:
nat_csv = os.path.join(RESULTS_DIR,'national_monthly_water_area.csv')

if not os.path.exists(nat_csv):
    print("[SKIPPED] Run Section 3 first to generate the water area CSV.")
else:
    nat = pd.read_csv(nat_csv)
    MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

    # ── Fig A: Seasonal ribbon ──────────────────────────────────────────────
    stats = nat.groupby('month')['water_area_km2'].agg(['mean','std','min','max']).sort_index()
    x = np.arange(12); mean_v = stats['mean'].values; std_v = stats['std'].values
    min_v = stats['min'].values; max_v = stats['max'].values

    fig, ax = plt.subplots(figsize=(11, 5))
    seasons = [(0,2,'#E8F4FD','Dry Winter'),(2,5,'#FFF8E1','Pre-Monsoon'),
               (5,9,'#FFEBEE','Monsoon'),(9,11,'#E8F5E9','Post-Monsoon')]
    ymax = max(max_v)*1.12
    for s,e,c,n in seasons:
        ax.axvspan(s-.5,e-.5,alpha=.12,color=c)
        ax.text((s+e)/2-.5,ymax*.97,n,ha='center',fontsize=8,fontstyle='italic',color='#666')
    ax.fill_between(x,min_v,max_v,alpha=.10,color='#1f77b4',label='Min-Max (11 yr)')
    ax.fill_between(x,mean_v-std_v,mean_v+std_v,alpha=.22,color='#1f77b4',label='Mean ± 1SD')
    ax.plot(x,mean_v,'o-',color='#1f77b4',lw=2.2,ms=7,mfc='white',mew=2,label='11-year Mean')
    ax.set_xticks(x); ax.set_xticklabels(MONTH_LABELS)
    ax.set_ylabel('Surface Water Area (km²)'); ax.set_xlabel('Month')
    ax.set_title('Mean Monthly Surface Water Area — Bangladesh (2015–2025)')
    ax.legend(loc='lower left'); ax.grid(True,alpha=.2,ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout(); fig.savefig(os.path.join(FIGURES_DIR,'fig_seasonal_ribbon.png'),dpi=300)
    plt.show(); print("✓ fig_seasonal_ribbon.png saved")

    # ── Fig B: July peak trend ──────────────────────────────────────────────
    july = nat[nat['month']==7].sort_values('year')
    yrs  = july['year'].values.astype(float)
    area = july['water_area_km2'].values.astype(float)
    slope,intercept,r,p,_ = scipy_stats.linregress(yrs,area)

    fig,ax = plt.subplots(figsize=(10,5))
    ax.scatter(yrs,area,color='#1f77b4',s=90,zorder=5,edgecolors='white',lw=1.5)
    ax.plot(yrs,area,'-',color='#1f77b4',alpha=.4,lw=1.5)
    ax.plot(yrs,slope*yrs+intercept,'--',color='#d62728',lw=2.5,
            label=f'Trend: {slope:+.1f} km²/yr  (R²={r**2:.3f}, p={p:.3f})')
    ax.set_xlabel('Year'); ax.set_ylabel('Peak Water Area — July (km²)')
    ax.set_title('Decadal Trend in Peak Monsoon Water Extent (July, 2015–2025)')
    ax.legend(); ax.grid(True,alpha=.2,ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout(); fig.savefig(os.path.join(FIGURES_DIR,'fig_july_trend.png'),dpi=300)
    plt.show(); print("✓ fig_july_trend.png saved")

    print("\n✓ All publication figures saved to figures/")
